# Sea-ice area, extent and volume against observations

Seasonal cycle of Southern Hemisphere sea-ice area, extent and
volume, model against observations.

The original pulls the NSIDC climate data record live from an
ERDDAP server. Here the observations come through ESMValTool's
CMORisers and are therefore just more `Dataset` objects. Run the
CMORisers first:

```
esmvaltool data format NSIDC-G02202-sh
```

**Area and extent both count only cells at or above 15%
concentration**, which is the NSIDC convention and the reason
extent is always at least the area. Weighting every cell
regardless of concentration, which a naive reading of "area"
suggests, breaks that relation as soon as the marginal ice zone is
wide.

---

Translated from the COSIMA recipe
[`Sea_Ice_Area_Concentration_Volume_with_Obs.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Sea_Ice_Area_Concentration_Volume_with_Obs.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
from esmvalcore.preprocessor import regrid

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
model = Dataset(short_name='siconc', project='CMIP6', mip='SImon',
                exp='historical', dataset='ACCESS-CM2',
                ensemble='r1i1p1f1', grid='gn',
                timerange='1991/2010')
model.add_supplementary(short_name='areacello', mip='Ofx')

obs = Dataset(short_name='siconc', project='OBS6',
              dataset='NSIDC-G02202-sh', mip='SImon',
              type='reanaly', version='4', tier=2,
              timerange='1991/2010')

datasets = {'ACCESS-CM2': model, 'NSIDC-G02202-sh': obs}
cubes = {}
for name, dataset in datasets.items():
    try:
        cubes[name] = regrid(dataset.load(), target_grid='1x1',
                             scheme='nearest')
    except Exception as error:
        print(f'{name} not available:', error)

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


ICE_THRESHOLD = 0.15


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def lat_2d(cube):
    """2D latitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("latitude")
    if coord.ndim == 2:
        return coord.points
    lon = cube.coord("longitude").points
    return np.broadcast_to(coord.points[:, None], (coord.shape[0], lon.size))


def guess_bounds(cube, coords=("latitude", "longitude")):
    """Add bounds in place where they are missing."""
    for name in coords:
        if cube.coords(name) and not cube.coord(name).has_bounds():
            try:
                cube.coord(name).guess_bounds()
            except ValueError:  # scalar or single-point coordinate
                pass
    return cube


def horizontal_slice(cube):
    """First slice of a cube over its two horizontal dimensions.

    ``cube.slices(["latitude", "longitude"])`` cannot be used here: on a
    curvilinear grid the two coordinates share both dimensions and iris
    rejects them as non-orthogonal.  Slicing by dimension index works
    for regular and curvilinear grids alike.
    """
    horizontal_dims = sorted(
        set(cube.coord_dims("latitude")) | set(cube.coord_dims("longitude")))
    if cube.ndim == len(horizontal_dims):
        return cube
    index = tuple(slice(None) if dim in horizontal_dims else 0
                  for dim in range(cube.ndim))
    return cube[index]


def spherical_polygon_area(lon_vertices, lat_vertices, radius=EARTH_RADIUS):
    """Area in m2 of spherical polygons given their corner coordinates.

    ``lon_vertices`` and ``lat_vertices`` are in degrees with the corners
    along the last axis, which is the ``(..., 4)`` layout CMOR uses for
    ``vertices_longitude`` / ``vertices_latitude``.  Any number of
    corners works as long as they are ordered around the cell.

    The area is the spherical excess, accumulated over a triangle fan
    from the first corner, using the Van Oosterom & Strackee (1983)
    form:

        tan(E/2) = |v1 . (v2 x v3)|
                   / (1 + v1.v2 + v2.v3 + v3.v1)

    with ``v`` the unit vectors of the corners.  This is *exact* for
    great-circle edges and therefore invariant under rotation of the
    sphere, which matters because a tripolar grid is exactly a rotated
    and stretched one: a formula that is only correct for cells aligned
    with latitude circles would degrade precisely where the grid stops
    being aligned.

    The obvious alternative, the spherical shoelace formula
    ``R^2/2 sum (dlon)(sin lat_i + sin lat_{i+1})``, treats each edge as
    linear in ``(lon, sin lat)`` rather than as a great circle.  It is
    exact for a latitude-longitude box and accurate to second order for
    small cells, but it is not rotation invariant and it fails badly for
    cells spanning large longitude ranges, such as those beside a
    displaced pole.

    The two disagree by about 2e-4 relative on a 2-degree cell, and
    neither is "wrong": they assume different cell edges.  A cell whose
    north and south edges follow *parallels* has the shoelace area,
    which is what ``iris.analysis.cartography.area_weights`` returns; a
    cell with great-circle edges has this one.  Both tile the sphere
    exactly.  The discrepancy is far smaller than the difference
    between either and the model's own ``areacello``, which is why
    ``areacello`` is still preferred when it is published.
    """
    lon = np.deg2rad(np.asarray(lon_vertices, dtype=float))
    lat = np.deg2rad(np.asarray(lat_vertices, dtype=float))
    unit = np.stack([np.cos(lat) * np.cos(lon),
                     np.cos(lat) * np.sin(lon),
                     np.sin(lat)], axis=-1)

    first = unit[..., 0, :]
    excess = np.zeros(unit.shape[:-2])
    for corner in range(1, unit.shape[-2] - 1):
        second = unit[..., corner, :]
        third = unit[..., corner + 1, :]
        triple = np.abs(np.einsum("...i,...i->...",
                                  first, np.cross(second, third)))
        denominator = (1.0
                       + np.einsum("...i,...i->...", first, second)
                       + np.einsum("...i,...i->...", second, third)
                       + np.einsum("...i,...i->...", third, first))
        excess += 2.0 * np.arctan2(triple, denominator)
    return excess * radius**2


def area_from_bounds(cube, radius=EARTH_RADIUS):
    """Cell area in m2 from 2D coordinate bounds, or ``None``.

    This is what makes a tripolar grid workable without ``areacello``.
    CMOR requires bounds on ``latitude`` and ``longitude``, and on a
    curvilinear grid those bounds are the four corner vertices of every
    cell, so the area follows exactly from
    :func:`spherical_polygon_area`.

    ESMValCore does not do this: ``try_adding_calculated_cell_area``
    handles regular and rotated-pole grids and raises
    ``CoordinateMultiDimError`` for everything else.  The information is
    in the file regardless.

    Returns ``None`` when the cube has no usable 2D bounds, so callers
    can fall back.
    """
    horizontal = horizontal_slice(cube)
    lat_coord = horizontal.coord("latitude")
    lon_coord = horizontal.coord("longitude")
    if lat_coord.ndim != 2 or lon_coord.ndim != 2:
        return None
    if lat_coord.bounds is None or lon_coord.bounds is None:
        return None
    if lat_coord.bounds.shape[-1] < 3:
        return None
    return np.ma.masked_invalid(
        spherical_polygon_area(lon_coord.bounds, lat_coord.bounds,
                               radius=radius))


def cell_area(cube):
    """Cell area in m2, preferring the model's own ``areacello``.

    Order of preference:

    1. an ``areacello`` cell measure attached to the cube, which is what
       ESMValCore builds from a ``supplementary_variables`` entry.  This
       is the model's own area and is always the best answer: it
       accounts for the actual grid generation, including any partial
       cells at the coast.
    2. a spherical polygon area from the 2D corner vertices, via
       :func:`area_from_bounds`.  CMOR mandates bounds on latitude and
       longitude, so this works on a tripolar grid with no
       ``areacello`` at all.  It assumes great-circle cell edges on a
       sphere, so it differs from the model's own areas by well under a
       percent for an ocean grid.
    3. spherical areas from 1D latitude/longitude bounds, for a regular
       grid.

    Only if all three fail does this raise, and then with instructions.
    """
    horizontal = horizontal_slice(cube)

    for measure in horizontal.cell_measures():
        if measure.var_name in ("areacello", "areacella", "areacell"):
            return np.ma.masked_invalid(
                np.ma.asarray(
                    horizontal.cell_measure(measure.name()).core_data(),
                    dtype=float).reshape(horizontal.shape))

    from_bounds = area_from_bounds(horizontal)
    if from_bounds is not None:
        return from_bounds

    if horizontal.coord("latitude").ndim > 1:
        raise ValueError(
            "this cube has 2D (curvilinear) latitude/longitude and no cell "
            "corner bounds, so no cell area can be obtained. Either attach "
            "areacello -- in a recipe with\n"
            "    supplementary_variables:\n"
            "      - short_name: areacello\n"
            "        mip: Ofx\n"
            "(add 'exp: piControl' if the experiment you want does not "
            "publish it), in a notebook with "
            "dataset.add_supplementary(short_name='areacello', mip='Ofx') "
            "-- or keep the coordinate bounds, which CMOR requires and "
            "which are enough to compute the area exactly.")

    # iris defaults to an Earth radius of 6367470 m where the cube has
    # no coordinate system, while cell_lengths uses the mean radius of
    # 6371000 m. That is a 0.11% difference in area, small but free to
    # avoid: pin the same sphere so lengths and areas agree.
    horizontal = horizontal.copy()
    guess_bounds(horizontal)
    if horizontal.coord("latitude").coord_system is None:
        sphere = iris.coord_systems.GeogCS(EARTH_RADIUS)
        horizontal.coord("latitude").coord_system = sphere
        horizontal.coord("longitude").coord_system = sphere
    return iris.analysis.cartography.area_weights(horizontal)


def fraction_from_percent(cube):
    """Sea-ice concentration as a fraction, whatever the input units.

    CMOR ``siconc`` is a percentage but several models publish a
    fraction.  The units are trusted when they are meaningful and the
    data range is used only as a fallback -- masking the fill values
    first, which is the bug that made the Russell sea-ice figures come
    out empty.
    """
    data = masked_data(cube)
    data = np.ma.masked_greater(data, 1.0e15)
    units = str(cube.units).strip().lower()
    if units in ("%", "percent"):
        return data / 100.0
    if units in ("1", "", "unknown", "none"):
        return data / 100.0 if np.ma.max(data) > 5.0 else data
    return data / 100.0 if np.ma.max(data) > 5.0 else data


def seaice_area(concentration, area, threshold=ICE_THRESHOLD,
                weighted=True):
    """Total sea-ice area or extent in m2, NSIDC conventions.

    Both quantities count only cells at or above ``threshold``:

    - *area* (``weighted=True``) is ``sum(concentration * cell_area)``;
    - *extent* (``weighted=False``) is ``sum(cell_area)``.

    Applying the threshold to the area as well as to the extent is what
    makes extent >= area, which is the relation these two numbers are
    always quoted with.  Weighting every cell regardless of
    concentration, as a naive reading of "area" suggests, breaks that
    relation as soon as the marginal ice zone is wide.

    Pass ``threshold=None`` to weight every cell, for the rare case
    where that is genuinely wanted.
    """
    conc = np.ma.masked_invalid(np.ma.asarray(concentration, dtype=float))
    area = np.ma.masked_invalid(np.ma.asarray(area, dtype=float))
    if threshold is not None:
        conc = np.ma.masked_where(conc < threshold, conc)
    if not weighted:
        return float(np.ma.sum(np.ma.masked_where(
            np.ma.getmaskarray(conc), area)))
    return float(np.ma.sum(conc * area))

## Seasonal cycle

`cell_area` prefers the `areacello` attached above and only falls
back to deriving an area from the coordinates on a regular grid.
That fallback is why this notebook regrids: on a native tripolar
grid `latitude` is 2-D, no 1-D area can be derived, and
`areacello` becomes mandatory rather than merely preferable.
Here the fields are intensive (a concentration, a volume per unit
area), so regridding them is legitimate — unlike the transports
in notebooks 02 and 12.

In [ ]:
M2_TO_MILLION_KM2 = 1.0e-12

def seasonal_cycle(cube, weighted=True):
    """Monthly climatology of area (weighted) or extent."""
    values = fraction_from_percent(cube)
    area = cell_area(cube)
    area = np.ma.masked_where(lat_2d(cube) >= 0, area)
    months = np.array([c.point.month
                       for c in cube.coord('time').cells()])

    totals = np.ma.masked_all(12)
    for month in range(1, 13):
        steps = np.where(months == month)[0]
        if steps.size:
            totals[month - 1] = np.mean([
                seaice_area(values[s], area, weighted=weighted)
                for s in steps])
    return totals * M2_TO_MILLION_KM2

results = {name: (seasonal_cycle(cube, weighted=True),
                  seasonal_cycle(cube, weighted=False))
           for name, cube in cubes.items()}

## Plot

In [ ]:
months = np.arange(1, 13)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, (area, extent) in results.items():
    is_obs = 'NSIDC' in name or 'OSI' in name
    style = dict(color='black' if is_obs else 'tab:blue',
                 linestyle='--' if is_obs else '-',
                 linewidth=2.2 if is_obs else 1.6, label=name)
    axes[0].plot(months, area, **style)
    axes[1].plot(months, extent, **style)

for ax, title in zip(axes, ['Sea-ice area', 'Sea-ice extent']):
    ax.set_xticks(months)
    ax.set_xlabel('Month')
    ax.set_ylabel('10$^6$ km$^2$')
    ax.set_title(f'{title} (Southern Hemisphere)')
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()